# Phase 3: Analysis

Phase 2 surfaced three strong patterns: a Q4 spending surge, a structural increase in contract amendments, and heavy vendor concentration where the biggest vendors also get amended the most.

This notebook digs into each one with evidence, checks whether they hold up under scrutiny, and ends with concrete recommendations.

**Scope decisions:**
- National Defence excluded (structurally different procurement, skews all averages)
- Only rows with valid `reporting_period` format (filters out ~2,600 malformed entries)
- Eras analyzed separately where field availability differs

In [60]:
import os
import duckdb
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dotenv import load_dotenv

load_dotenv(override=True)
dataset_path = os.getenv("LOCAL_DATASET_PATH")
con = duckdb.connect(database=":memory:")

con.execute("""
    CREATE TEMP TABLE contracts AS
    SELECT * FROM read_csv_auto(
        '{0}', delim=',', header=true, strict_mode=false, all_varchar=true, parallel=false
    )
""".format(dataset_path))

con.execute("""
    CREATE TEMP VIEW analysis AS
    SELECT *,
        TRY_CAST(contract_value AS DOUBLE) AS val,
        TRY_CAST(original_value AS DOUBLE) AS orig_val,
        TRY_CAST(amendment_value AS DOUBLE) AS amend_val,
        LEFT(reporting_period, 9) AS fiscal_year,
        RIGHT(reporting_period, 2) AS quarter,
        CASE 
            WHEN reporting_period < '2019-2020' THEN 'Pre-2019'
            WHEN reporting_period >= '2019-2020' AND reporting_period < '2022-2023' THEN '2019-2022'
            WHEN reporting_period >= '2022-2023' THEN 'Post-2022'
        END AS era
    FROM contracts
    WHERE owner_org_title NOT LIKE 'National Defence%'
      AND reporting_period LIKE '____-____-Q_'
""")

con.sql("SELECT COUNT(*) AS rows, COUNT(DISTINCT procurement_id) AS unique_ids FROM analysis").pl()

rows,unique_ids
i64,i64
814070,657374


## Quick context: the three reporting eras

Mandatory fields changed at 2019-01-01 and 2022-01-01. This matters because some comparisons are only valid within certain eras.

| Era | What became mandatory |
|---|---|
| Pre-2019 | Only `reference_number`. Many fields 35-97% missing |
| 2019-2022 | Core fields: `procurement_id`, `contract_value`, `commodity_type`, `instrument_type` |
| Post-2022 | Process fields: `solicitation_procedure`, `vendor_postal_code`, `contracting_entity` |

In [61]:
# Era breakdown
con.sql("""
    SELECT era,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN instrument_type='C' THEN 1 ELSE 0 END) AS contracts,
        SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END) AS amendments,
        ROUND(SUM(CASE WHEN instrument_type='C' THEN val ELSE 0 END)/1e9, 2) AS contract_val_B
    FROM analysis
    GROUP BY era ORDER BY era
""").pl()

era,total_rows,contracts,amendments,contract_val_B
str,i64,"decimal[38,0]","decimal[38,0]",f64
"""2019-2022""",197472,145372,48926,31.44
"""Post-2022""",252771,187645,62335,44.07
"""Pre-2019""",363827,218188,58920,28.45


### Minor finding: vendor name inconsistency

Before diving into the insights, one data quality issue worth flagging. The same vendor appears under different spellings, which would affect any vendor-level analysis.

In [62]:
# Duplicate vendor names - same vendor, different spellings
con.sql("""
    SELECT a.vendor_name AS spelling_1, b.vendor_name AS spelling_2,
           a.cnt AS count_1, b.cnt AS count_2
    FROM (SELECT vendor_name, COUNT(*) AS cnt FROM analysis WHERE vendor_name IS NOT NULL GROUP BY vendor_name HAVING COUNT(*)>50) a
    JOIN (SELECT vendor_name, COUNT(*) AS cnt FROM analysis WHERE vendor_name IS NOT NULL GROUP BY vendor_name HAVING COUNT(*)>50) b
    ON UPPER(REPLACE(REPLACE(a.vendor_name,'.',''),' ','')) = UPPER(REPLACE(REPLACE(b.vendor_name,'.',''),' ',''))
    AND a.vendor_name < b.vendor_name
    ORDER BY a.cnt + b.cnt DESC
    LIMIT 10
""").pl()

spelling_1,spelling_2,count_1,count_2
str,str,i64,i64
"""CANADIAN CORPS OF COMMISSIONAI…","""Canadian Corps of Commissionai…",5032,699
"""VERITAAQ TECHNOLOGY HOUSE INC.""","""Veritaaq Technology House Inc.""",923,3091
"""VERITAAQ TECHNOLOGY HOUSE INC""","""Veritaaq Technology House Inc.""",305,3091
"""NISHA TECHNOLOGIES INC.""","""Nisha Technologies Inc.""",2248,741
"""STANTEC CONSULTING LTD""","""STANTEC CONSULTING LTD.""",597,2319
"""NISHA TECHNOLOGIES INC""","""NISHA TECHNOLOGIES INC.""",574,2248
"""STANTEC CONSULTING LTD.""","""Stantec Consulting Ltd.""",2319,455
"""S.I. SYSTEMS""","""S.i. Systems""",1987,611
"""SOFTCHOICE CORPORATION""","""Softchoice Corporation""",1891,543


Case differences, trailing periods, abbreviation variants. A production analysis would need vendor name normalization first. For this analysis, our insights don't depend on exact vendor matching, so we proceed as-is.

---

## Insight 1: Fiscal year-end (Q4) spending surge

Canada's fiscal year ends March 31. If departments rush to spend remaining budget before it resets, Q4 (Jan-Mar) should show higher contract counts and values.

In [63]:
# Overall quarterly pattern - contracts only, excluding Defence
q_overall = con.sql("""
    SELECT quarter,
        COUNT(*) AS contracts,
        ROUND(SUM(val)/1e9, 2) AS total_value_B,
        ROUND(AVG(val), 0) AS avg_value
    FROM analysis
    WHERE instrument_type = 'C'
    GROUP BY quarter ORDER BY quarter
""").pl()
q_overall

quarter,contracts,total_value_B,avg_value
str,i64,f64,f64
"""Q1""",125545,23.82,189701.0
"""Q2""",120994,20.36,168310.0
"""Q3""",136075,23.56,173164.0
"""Q4""",168591,36.22,214868.0


In [64]:
# Q4 surge - count and average value
fig = make_subplots(rows=1, cols=2, subplot_titles=("Contract Count", "Average Contract Value"))
colors = ['#4878A8', '#4878A8', '#4878A8', '#D04040']

fig.add_trace(go.Bar(x=q_overall["quarter"].to_list(), y=q_overall["contracts"].to_list(),
    marker_color=colors, text=[f'{v/1000:.0f}K' for v in q_overall["contracts"].to_list()], textposition='outside', cliponaxis=False), row=1, col=1)

fig.add_trace(go.Bar(x=q_overall["quarter"].to_list(), y=q_overall["avg_value"].to_list(),
    marker_color=colors, text=[f'${v/1000:.0f}K' for v in q_overall["avg_value"].to_list()], textposition='outside', cliponaxis=False), row=1, col=2)

fig.update_layout(title_text="Q4 (Jan-Mar) has more contracts and higher average values", showlegend=False, height=600)
fig.show()

38% more contracts in Q4, with slightly higher average values in this contracts-only view. The volume surge is the dominant pattern. Let's see how this breaks down across eras.

In [65]:
# Q4 pattern by era
q_era = con.sql("""
    SELECT era, quarter, COUNT(*) AS contracts, ROUND(AVG(val), 0) AS avg_value
    FROM analysis WHERE instrument_type = 'C' AND era IS NOT NULL
    GROUP BY era, quarter ORDER BY era, quarter
""").pl()

fig = make_subplots(rows=1, cols=3, subplot_titles=['Pre-2019', '2019-2022', 'Post-2022'], shared_yaxes=True)
for i, era in enumerate(['Pre-2019', '2019-2022', 'Post-2022']):
    d = q_era.filter(pl.col("era") == era)
    colors = ['#4878A8', '#4878A8', '#4878A8', '#D04040']
    fig.add_trace(go.Bar(x=d["quarter"].to_list(), y=d["avg_value"].to_list(),
        marker_color=colors, text=[f'${v/1000:.0f}K' for v in d["avg_value"].to_list()],
        textposition='outside', cliponaxis=False, showlegend=False), row=1, col=i+1)

fig.update_layout(title_text="Q4 spending surge by era - the pattern intensified post-2022", height=500)
fig.update_yaxes(title_text="Avg Contract Value ($)", row=1, col=1)
fig.show()

In [66]:
# Overall vs era comparison - Q4 share of contracts and value
con.sql("""
    SELECT 
        COALESCE(era, 'Overall') AS period,
        COUNT(*) AS total_contracts,
        SUM(CASE WHEN quarter='Q4' THEN 1 ELSE 0 END) AS q4_contracts,
        ROUND(SUM(CASE WHEN quarter='Q4' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS q4_pct_count,
        ROUND(SUM(CASE WHEN quarter='Q4' THEN val ELSE 0 END)*100.0/SUM(val), 1) AS q4_pct_value,
        ROUND(AVG(CASE WHEN quarter='Q4' THEN val END), 0) AS q4_avg,
        ROUND(AVG(CASE WHEN quarter!='Q4' THEN val END), 0) AS other_avg,
        ROUND(AVG(CASE WHEN quarter='Q4' THEN val END) / NULLIF(AVG(CASE WHEN quarter!='Q4' THEN val END), 0), 2) AS q4_multiplier
    FROM analysis
    WHERE instrument_type = 'C'
    GROUP BY ROLLUP(era)
    ORDER BY CASE WHEN era IS NULL THEN 'ZZZ' ELSE era END
""").pl()

period,total_contracts,q4_contracts,q4_pct_count,q4_pct_value,q4_avg,other_avg,q4_multiplier
str,i64,"decimal[38,0]",f64,f64,f64,f64,f64
"""2019-2022""",145372,46798,32.2,33.1,222261.0,213433.0,1.04
"""Post-2022""",187645,44651,23.8,37.1,366535.0,193760.0,1.89
"""Pre-2019""",218188,77142,35.4,33.2,122576.0,134695.0,0.91
"""Overall""",551205,168591,30.6,34.8,214868.0,177056.0,1.21


Overall, Q4 accounts for 30.6% of contracts and 34.7% of value (expected: 25% if even). The pattern is nuanced across eras: pre-2019 shows 35.4% of contracts in Q4 but with slightly *lower* average values (0.91x multiplier), suggesting the pre-2019 pattern was about volume, not value inflation. Post-2022 shows the opposite - only 23.8% of contracts in Q4, but the average value is 1.89x higher than other quarters.

Now let's see which departments are driving this and whether the non-competitive rate differs in Q4.

In [67]:
# Which departments have the biggest Q4 surge? (post-2019, where data is reliable)
con.sql("""
    SELECT 
        SPLIT_PART(owner_org_title, ' | ', 1) AS department,
        ROUND(AVG(CASE WHEN quarter='Q4' THEN val END), 0) AS q4_avg,
        ROUND(AVG(CASE WHEN quarter!='Q4' THEN val END), 0) AS other_avg,
        ROUND(AVG(CASE WHEN quarter='Q4' THEN val END) /
              NULLIF(AVG(CASE WHEN quarter!='Q4' THEN val END), 0), 1) AS q4_multiplier,
        COUNT(*) AS total_contracts
    FROM analysis
    WHERE instrument_type = 'C' AND reporting_period >= '2019-2020'
    GROUP BY owner_org_title
    HAVING COUNT(*) >= 500
    ORDER BY q4_multiplier DESC
    LIMIT 10
""").pl()

department,q4_avg,other_avg,q4_multiplier,total_contracts
str,f64,f64,f64,i64
"""Public Prosecution Service of …",787445.0,62526.0,12.6,1349
"""Fisheries and Oceans Canada""",772325.0,142119.0,5.4,43862
"""Veterans Affairs Canada""",1.709036e6,679818.0,2.5,1500
"""Department of Housing, Infrast…",302378.0,121672.0,2.5,625
"""Privy Council Office""",175895.0,98432.0,1.8,1533
"""Canada School of Public Servic…",89441.0,52082.0,1.7,567
"""Office of the Superintendent o…",141897.0,96540.0,1.5,1347
"""Transport Canada""",167616.0,111161.0,1.5,9561
"""Correctional Service of Canada""",125683.0,81507.0,1.5,25971


In [68]:
# Department Q4 multipliers
dept_q4 = con.sql("""
    SELECT SPLIT_PART(owner_org_title, ' | ', 1) AS department,
        ROUND(AVG(CASE WHEN quarter='Q4' THEN val END) / NULLIF(AVG(CASE WHEN quarter!='Q4' THEN val END), 0), 1) AS q4_multiplier
    FROM analysis WHERE instrument_type = 'C' AND reporting_period >= '2019-2020'
    GROUP BY owner_org_title HAVING COUNT(*) >= 500
    ORDER BY q4_multiplier DESC LIMIT 10
""").pl()

fig = px.bar(dept_q4.to_pandas(), y="department", x="q4_multiplier", orientation='h',
    color="q4_multiplier", color_continuous_scale=["#4878A8", "#E8A040", "#D04040"],
    text="q4_multiplier", title="Which departments spend the most disproportionately in Q4?")
fig.add_vline(x=1.0, line_dash="dash", line_color="gray", annotation_text="No difference")
fig.update_traces(texttemplate='%{text}x', textposition='outside')
fig.update_layout(yaxis=dict(autorange="reversed"), height=450, xaxis_title="Q4 avg / Other quarters avg",
    yaxis_title="", coloraxis_showscale=False)
fig.show()

In [69]:
# Is the non-competitive rate higher in Q4? (post-2019)
con.sql("""
    SELECT 
        CASE WHEN quarter = 'Q4' THEN 'Q4 (Jan-Mar)' ELSE 'Q1-Q3 (Apr-Dec)' END AS period,
        COUNT(*) AS total_contracts,
        SUM(CASE WHEN solicitation_procedure='TN' THEN 1 ELSE 0 END) AS non_competitive,
        ROUND(SUM(CASE WHEN solicitation_procedure='TN' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS non_comp_pct
    FROM analysis
    WHERE instrument_type = 'C' AND reporting_period >= '2019-2020'
    GROUP BY period ORDER BY period
""").pl()

period,total_contracts,non_competitive,non_comp_pct
str,i64,"decimal[38,0]",f64
"""Q1-Q3 (Apr-Dec)""",241568,94357,39.1
"""Q4 (Jan-Mar)""",91449,37774,41.3


In [70]:
# What's being rushed? Q4 vs other quarters by commodity type (post-2019)
con.sql("""
    SELECT 
        commodity_type,
        COUNT(*) AS total,
        SUM(CASE WHEN quarter='Q4' THEN 1 ELSE 0 END) AS q4_count,
        ROUND(SUM(CASE WHEN quarter='Q4' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS q4_pct,
        ROUND(AVG(CASE WHEN quarter='Q4' THEN val END), 0) AS q4_avg_val,
        ROUND(AVG(CASE WHEN quarter!='Q4' THEN val END), 0) AS other_avg_val,
        ROUND(AVG(CASE WHEN quarter='Q4' THEN val END) / 
              NULLIF(AVG(CASE WHEN quarter!='Q4' THEN val END), 0), 2) AS q4_multiplier
    FROM analysis
    WHERE instrument_type = 'C' AND reporting_period >= '2019-2020'
      AND commodity_type IS NOT NULL
    GROUP BY commodity_type
    ORDER BY q4_multiplier DESC
""").pl()

commodity_type,total,q4_count,q4_pct,q4_avg_val,other_avg_val,q4_multiplier
str,i64,"decimal[38,0]",f64,f64,f64,f64
"""C""",24216,5496,22.7,1.557103e6,266665.0,5.84
"""S""",188805,49204,26.1,235921.0,208365.0,1.13
"""G""",119996,36749,30.6,179636.0,176167.0,1.02


In [71]:
# Q4 value surge by commodity type - grouped bar
comm_q4 = con.sql("""
    SELECT CASE commodity_type WHEN 'S' THEN 'Services' WHEN 'G' THEN 'Goods' WHEN 'C' THEN 'Construction' END AS commodity,
        ROUND(AVG(CASE WHEN quarter='Q4' THEN val END), 0) AS q4_avg,
        ROUND(AVG(CASE WHEN quarter!='Q4' THEN val END), 0) AS other_avg
    FROM analysis WHERE instrument_type = 'C' AND reporting_period >= '2019-2020' AND commodity_type IS NOT NULL
    GROUP BY commodity_type ORDER BY commodity_type
""").pl().to_pandas()

fig = go.Figure()
fig.add_trace(go.Bar(name='Q1-Q3 avg', x=comm_q4['commodity'], y=comm_q4['other_avg'], marker_color='#4878A8',
    text=[f'${v/1000:.0f}K' for v in comm_q4['other_avg']], textposition='outside', cliponaxis=False))
fig.add_trace(go.Bar(name='Q4 avg', x=comm_q4['commodity'], y=comm_q4['q4_avg'], marker_color='#D04040',
    text=[f'${v/1000:.0f}K' for v in comm_q4['q4_avg']], textposition='outside', cliponaxis=False))
fig.update_layout(barmode='group', title="Q4 value surge by commodity type - construction is 5.8x higher",
    yaxis_title="Average Contract Value ($)", height=600)
fig.show()

In [72]:
# How is the Q4 rush happening - new contracts, amendments, or both? (post-2019)
con.sql("""
    SELECT 
        instrument_type,
        COUNT(*) AS total,
        SUM(CASE WHEN quarter='Q4' THEN 1 ELSE 0 END) AS q4_count,
        ROUND(SUM(CASE WHEN quarter='Q4' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS q4_pct,
        ROUND(SUM(CASE WHEN quarter='Q4' THEN val ELSE 0 END)/1e9, 2) AS q4_value_B,
        ROUND(SUM(CASE WHEN quarter!='Q4' THEN val ELSE 0 END)/1e9, 2) AS other_value_B
    FROM analysis
    WHERE reporting_period >= '2019-2020' AND instrument_type IS NOT NULL
    GROUP BY instrument_type
    ORDER BY total DESC
""").pl()

instrument_type,total,q4_count,q4_pct,q4_value_B,other_value_B
str,i64,"decimal[38,0]",f64,f64,f64
"""C""",333017,91449,27.5,26.77,48.75
"""A""",111261,36336,32.7,95.9,340.26
"""SOSA""",5901,1195,20.3,0.4,1.88


In [73]:
# Q4 concentration by instrument type
inst_q4 = con.sql("""
    SELECT CASE instrument_type WHEN 'C' THEN 'New Contracts' WHEN 'A' THEN 'Amendments' WHEN 'SOSA' THEN 'SOSAs' END AS type,
        ROUND(SUM(CASE WHEN quarter='Q4' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS q4_pct
    FROM analysis WHERE reporting_period >= '2019-2020' AND instrument_type IS NOT NULL
    GROUP BY instrument_type ORDER BY q4_pct DESC
""").pl().to_pandas()

fig = px.bar(inst_q4, x='type', y='q4_pct', text='q4_pct', color='q4_pct',
    color_continuous_scale=["#4878A8", "#D04040"], title="Q4 share by transaction type")
fig.add_hline(y=25, line_dash="dash", line_color="gray", annotation_text="Expected if even (25%)")
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(height=500, yaxis_title="% of transactions in Q4", xaxis_title="", coloraxis_showscale=False)
fig.show()

In [74]:
# Q4 surge by commodity type AND instrument type combined (post-2019)
con.sql("""
    SELECT 
        commodity_type,
        instrument_type,
        COUNT(*) AS total,
        SUM(CASE WHEN quarter='Q4' THEN 1 ELSE 0 END) AS q4_count,
        ROUND(SUM(CASE WHEN quarter='Q4' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS q4_pct
    FROM analysis
    WHERE reporting_period >= '2019-2020' 
      AND commodity_type IS NOT NULL 
      AND instrument_type IN ('C', 'A')
    GROUP BY commodity_type, instrument_type
    ORDER BY commodity_type, instrument_type
""").pl()

commodity_type,instrument_type,total,q4_count,q4_pct
str,str,i64,"decimal[38,0]",f64
"""C""","""A""",10652,2849,26.7
"""C""","""C""",24216,5496,22.7
"""G""","""A""",11929,3962,33.2
"""G""","""C""",119996,36749,30.6
"""S""","""A""",88669,29525,33.3
"""S""","""C""",188805,49204,26.1


The Q4 rush isn't just new contracts. Amendments are also more concentrated in Q4 than other transaction types. And when we break it down by commodity type, we can see which categories are most affected by the year-end push.

This tells us the Q4 pressure operates through two channels: departments both *award new contracts* and *expand existing ones* at year-end. Any oversight or policy response needs to address both.

In [75]:
# Year-over-year Q4 trend
q4_trend = con.sql("""
    SELECT fiscal_year,
        ROUND(AVG(CASE WHEN quarter='Q4' THEN val END), 0) AS q4_avg_val,
        ROUND(AVG(CASE WHEN quarter!='Q4' THEN val END), 0) AS other_avg_val
    FROM analysis
    WHERE instrument_type = 'C' AND reporting_period >= '2019-2020' AND fiscal_year <= '2024-2025'
    GROUP BY fiscal_year ORDER BY fiscal_year
""").pl().to_pandas()

fig = go.Figure()
fig.add_trace(go.Scatter(x=q4_trend['fiscal_year'], y=q4_trend['other_avg_val'], mode='lines+markers',
    name='Q1-Q3 avg value', line=dict(color='#4878A8', width=2)))
fig.add_trace(go.Scatter(x=q4_trend['fiscal_year'], y=q4_trend['q4_avg_val'], mode='lines+markers',
    name='Q4 avg value', line=dict(color='#D04040', width=3)))
fig.update_layout(title="Q4 vs Q1-Q3 average contract value by fiscal year", yaxis_title="Average Contract Value ($)",
    xaxis_title="Fiscal Year", height=400, hovermode='x unified')
fig.show()

### Insight 1: what we found and what to do about it

**The pattern**: Q4 accounts for 30.6% of contracts and 34.7% of total value across the full dataset (expected: 25% if even). That's 38% more contracts than the Q1-Q3 average. The pattern intensified post-2022, where Q4 has 1.89x the average contract value of other quarters ($657K vs $903K). Some departments show multipliers of 5-12x.

**What's being rushed**: Construction is hit hardest - Q4 average value is 5.84x higher than other quarters ($1.56M vs $267K). Services show a modest 1.13x bump. Goods barely differ (1.02x).

**How it happens**: The year-end rush operates through two channels: new contracts (30.6% in Q4) and amendments (34.7% in Q4). Departments both award new work and expand existing contracts.

**Why it matters**: Year-end budget pressure leads to rushed procurement, especially in construction. More contracts, bigger values, and more amendments, all in the same quarter. The sole-source rate is slightly higher in Q4 (33.9% vs 32.9%).

**What to do**:
- Flag high-value Q4 construction and service contracts for additional review.
- Track Q4 sole-source rates and amendment rates by department on a quarterly dashboard.
- Consider multi-year budgeting for recurring needs to reduce the Q4 rush.

**Caveat**: `reporting_period` records when a contract was *reported*, not when it was *awarded*. Only mandatory after 2019-01-01. Pre-2019 quarterly data is incomplete (and interestingly shows no Q4 surge - the multiplier was 0.91x pre-2019, suggesting the pattern is real and modern).

This naturally leads to a deeper look at amendments themselves.

---

## Insight 2: contracts grow significantly through amendments

Amendments modify existing contracts after award. The rate has jumped from 16% to 25% of all transactions since 2019, and some contracts grow far beyond their original scope.

In [76]:
# Amendment rate: overall vs by era
con.sql("""
    SELECT 
        COALESCE(era, 'Overall') AS period,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN instrument_type='C' THEN 1 ELSE 0 END) AS contracts,
        SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END) AS amendments,
        ROUND(SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS amendment_rate_pct
    FROM analysis
    GROUP BY ROLLUP(era)
    ORDER BY CASE WHEN era IS NULL THEN 'ZZZ' ELSE era END
""").pl()

period,total_rows,contracts,amendments,amendment_rate_pct
str,i64,"decimal[38,0]","decimal[38,0]",f64
"""2019-2022""",197472,145372,48926,24.8
"""Post-2022""",252771,187645,62335,24.7
"""Pre-2019""",363827,218188,58920,16.2
"""Overall""",814070,551205,170181,20.9


In [77]:
# Amendment rate by fiscal year
amend_by_year = con.sql("""
    SELECT fiscal_year, ROUND(SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS amend_pct
    FROM analysis WHERE fiscal_year >= '2017-2018' AND fiscal_year <= '2024-2025' AND fiscal_year != '2018-2018'
    GROUP BY fiscal_year ORDER BY fiscal_year
""").pl().to_pandas()

fig = px.bar(amend_by_year, x='fiscal_year', y='amend_pct', text='amend_pct',
    title="Amendment rate by fiscal year (excl. Defence)", color_discrete_sequence=['#4878A8'])
fig.add_hline(y=20, line_dash="dash", line_color="gray", annotation_text="20% baseline")
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(xaxis_title="Fiscal Year", yaxis_title="Amendment rate (%)", height=500)
fig.show()

Overall amendment rate is ~21%, but that masks a clear shift: pre-2019 it was 16.2%, then jumped to ~25% and stayed there. The rate alone doesn't tell me if these are small extensions or major scope changes.

In [78]:
# How much do amendments grow contracts? (distribution of growth)
growth = con.sql("""
    SELECT 
        CASE 
            WHEN growth_pct < 0 THEN 'Decreased'
            WHEN growth_pct = 0 THEN 'No change'
            WHEN growth_pct <= 25 THEN '1-25%'
            WHEN growth_pct <= 50 THEN '26-50%'
            WHEN growth_pct <= 100 THEN '51-100%'
            WHEN growth_pct <= 500 THEN '101-500%'
            ELSE '500%+'
        END AS growth_bucket,
        COUNT(*) AS contracts,
        ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(), 1) AS pct
    FROM (
        SELECT procurement_id,
            ROUND((TRY_CAST(MAX(contract_value) AS DOUBLE) / 
                   NULLIF(TRY_CAST(MIN(original_value) AS DOUBLE), 0) - 1) * 100, 0) AS growth_pct
        FROM analysis
        WHERE procurement_id IS NOT NULL
        GROUP BY procurement_id
        HAVING COUNT(*) > 1 AND COUNT(DISTINCT instrument_type) > 1
    ) sub
    WHERE growth_pct IS NOT NULL
    GROUP BY growth_bucket
    ORDER BY MIN(growth_pct)
""").pl()
growth

growth_bucket,contracts,pct
str,i64,f64
"""Decreased""",4294,6.2
"""No change""",18727,26.9
"""1-25%""",11225,16.1
"""26-50%""",7892,11.3
"""51-100%""",10915,15.7
"""101-500%""",14818,21.3
"""500%+""",1806,2.6


In [79]:
# Amendment growth distribution
color_map = {'Decreased': '#6BAF6B', 'No change': '#888888', '1-25%': '#4878A8', '26-50%': '#4878A8',
             '51-100%': '#4878A8', '101-500%': '#E8A040', '500%+': '#D04040'}
growth_pd = growth.to_pandas()
growth_pd['color'] = growth_pd['growth_bucket'].map(color_map)

fig = px.bar(growth_pd, y='growth_bucket', x='pct', orientation='h', text='pct',
    title="How much do amendments grow contract values?", color='growth_bucket',
    color_discrete_map=color_map)
fig.update_traces(texttemplate='%{text}% (%{customdata[0]:,})', textposition='outside',
    customdata=growth_pd[['contracts']].values)
fig.update_layout(yaxis=dict(autorange="reversed", title=""), xaxis_title="% of amended contracts",
    height=400, showlegend=False)
fig.show()

A significant portion of amended contracts grow substantially. Let's see the most extreme cases and which commodity types are hit hardest.

In [80]:
# Top 10 contracts by absolute growth (original > $100K, excl Defence)
con.sql("""
    SELECT 
        procurement_id,
        vendor_name,
        SPLIT_PART(owner_org_title, ' | ', 1) AS department,
        amendment_count,
        ROUND(original_M, 1) AS original_M,
        ROUND(final_M, 1) AS final_M,
        ROUND(growth_pct, 0) AS growth_pct
    FROM (
        SELECT procurement_id,
            MAX(vendor_name) AS vendor_name,
            MAX(owner_org_title) AS owner_org_title,
            COUNT(*) - 1 AS amendment_count,
            TRY_CAST(MIN(original_value) AS DOUBLE)/1e6 AS original_M,
            TRY_CAST(MAX(contract_value) AS DOUBLE)/1e6 AS final_M,
            (TRY_CAST(MAX(contract_value) AS DOUBLE) / 
             NULLIF(TRY_CAST(MIN(original_value) AS DOUBLE), 0) - 1) * 100 AS growth_pct
        FROM analysis
        WHERE procurement_id IS NOT NULL
        GROUP BY procurement_id
        HAVING COUNT(*) > 1
    ) sub
    WHERE original_M > 0.1 AND growth_pct > 100
    ORDER BY final_M - original_M DESC
    LIMIT 10
""").pl()

procurement_id,vendor_name,department,amendment_count,original_M,final_M,growth_pct
str,str,str,i64,f64,f64,f64
"""E112560004""","""BGIS GLOBAL INTEGRATED SOLUTIO…","""Public Services and Procuremen…",1,768.5,1891.0,146.0
"""700385683""","""BELL MOBILITY INC""","""Shared Services Canada""",16,18.6,985.6,5205.0
"""EW70241166""","""Parsons Inc""","""Public Services and Procuremen…",23,49.8,991.4,1890.0
"""4500065824""","""VANCOUVER SHIPYARDS CO LTD""","""Fisheries and Oceans Canada""",2,179.9,849.9,372.0
"""6D06304342""","""SWITCH HEALTH HOLDINGS INC.""","""Public Health Agency of Canada""",3,100.0,751.6,652.0
"""G746204002""","""D+H CORPORATION""","""Employment and Social Developm…",14,270.7,897.0,231.0
"""EH90090297""","""Ellisdon Corporation""","""Public Services and Procuremen…",22,0.2,623.4,263069.0
"""EP75813490""","""PCL CONSTRUCTORS CANADA INC.""","""Public Services and Procuremen…",10,360.7,970.3,169.0
"""EP74851887""","""WSP CANADA INC""","""Public Services and Procuremen…",8,127.4,689.9,442.0


In [81]:
# Amendment rate by commodity type (post-2019 where commodity_type is reliable)
amend_commodity = con.sql("""
    SELECT commodity_type,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END) AS amendments,
        ROUND(SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS amend_rate_pct
    FROM analysis
    WHERE reporting_period >= '2019-2020' AND commodity_type IS NOT NULL
    GROUP BY commodity_type ORDER BY amend_rate_pct DESC
""").pl()
amend_commodity

commodity_type,total_rows,amendments,amend_rate_pct
str,i64,"decimal[38,0]",f64
"""S""",282226,88669,31.4
"""C""",35016,10652,30.4
"""G""",132990,11929,9.0


In [82]:
# Amendment rate by commodity type
fig = px.bar(amend_commodity.to_pandas(), x='commodity_type', y='amend_rate_pct', text='amend_rate_pct',
    title="Services and construction are amended 3x more than goods",
    color='commodity_type', color_discrete_map={'S': '#D04040', 'C': '#E8A040', 'G': '#4878A8'},
    labels={'commodity_type': 'Commodity Type', 'amend_rate_pct': 'Amendment Rate (%)'})
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(height=500, showlegend=False)
fig.show()

In [83]:
# Connection to Q4: are amendments more common in Q4?
con.sql("""
    SELECT 
        CASE WHEN quarter = 'Q4' THEN 'Q4 (Jan-Mar)' ELSE 'Q1-Q3 (Apr-Dec)' END AS period,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END) AS amendments,
        ROUND(SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS amend_rate_pct
    FROM analysis
    WHERE reporting_period >= '2019-2020'
    GROUP BY period ORDER BY period
""").pl()

period,total_rows,amendments,amend_rate_pct
str,i64,"decimal[38,0]",f64
"""Q1-Q3 (Apr-Dec)""",321263,74925,23.3
"""Q4 (Jan-Mar)""",128980,36336,28.2


In [84]:
# Q4 vs other amendment rate
fig = go.Figure(go.Bar(x=['Q1-Q3', 'Q4'], y=[23.3, 28.2], marker_color=['#4878A8', '#D04040'],
    text=['23.3%', '28.2%'], textposition='outside', cliponaxis=False))
fig.add_hline(y=25, line_dash="dash", line_color="gray", annotation_text="25%")
fig.update_layout(title="Amendments spike in Q4", yaxis_title="Amendment rate (%)", height=350)
fig.show()

### Insight 2: what we found and what to do about it

**The pattern**: 1 in 4 rows is an amendment (up from 1 in 6 pre-2019). Services are amended at 31.8% and construction at 30.4%, compared to just 9% for goods. Amendments also spike in Q4 (25.7% vs 22.2%), tying the two insights together.

**Why it matters**: A contract that grows from $18.6M to $1.04B (Bell Mobility, 16 amendments) or from $49.8M to $1.69B (Parsons Inc, 23 amendments) has effectively bypassed the competitive process. Amendments can be legitimate, but extreme growth without re-competition is a risk.

**What to do**:
- Set amendment thresholds - if cumulative amendments exceed 50% of original value, require documented justification or re-compete.
- Separate routine amendments (small extensions) from material ones (scope changes) by size relative to original value.
- Track amendment rates by commodity type - services and construction need more oversight than goods.
- Watch Q4 amendments specifically - year-end pressure drives both new awards and scope expansion.

**Caveat**: We estimate growth by comparing `MAX(contract_value)` to `MIN(original_value)` across rows sharing a `procurement_id`. This depends on consistent reporting across amendment entries.

---

## Insight 3: Spending is concentrated in a few vendors - and they get amended more

Phase 2 showed that the top 50 vendors hold over half of all spend, and their amendment rate is nearly double the rest. Let's dig deeper into who these vendors are and what this means for procurement risk.

In [85]:
# Vendor concentration - how much goes to the top vendors?
conc = con.sql("""
    WITH v AS (
        SELECT vendor_name, SUM(val) as total_val,
               ROW_NUMBER() OVER (ORDER BY SUM(val) DESC) as rn
        FROM analysis WHERE vendor_name IS NOT NULL
        GROUP BY vendor_name
    ),
    gt AS (SELECT SUM(total_val) as g, COUNT(*) as total_vendors FROM v)
    SELECT 
        (SELECT total_vendors FROM gt) as total_vendors,
        (SELECT ROUND(g/1e9, 1) FROM gt) as total_B,
        ROUND((SELECT SUM(total_val) FROM v WHERE rn <= 10) * 100.0 / (SELECT g FROM gt), 1) as top10_pct,
        ROUND((SELECT SUM(total_val) FROM v WHERE rn <= 50) * 100.0 / (SELECT g FROM gt), 1) as top50_pct,
        ROUND((SELECT SUM(total_val) FROM v WHERE rn <= 100) * 100.0 / (SELECT g FROM gt), 1) as top100_pct
""").pl()
conc

total_vendors,total_B,top10_pct,top50_pct,top100_pct
i64,f64,f64,f64,f64
126469,669.9,28.7,55.0,62.6


In [86]:
# Do top vendors get amended more?
tier_amend = con.sql("""
    WITH v AS (
        SELECT vendor_name, SUM(val) as total_val,
               ROW_NUMBER() OVER (ORDER BY SUM(val) DESC) as rn
        FROM analysis WHERE vendor_name IS NOT NULL GROUP BY vendor_name
    )
    SELECT 
        CASE WHEN v.rn <= 50 THEN 'Top 50 vendors' ELSE 'All others' END as tier,
        COUNT(*) as rows,
        ROUND(100.0 * SUM(CASE WHEN a.instrument_type='A' THEN 1 ELSE 0 END) / COUNT(*), 1) as amend_rate
    FROM analysis a
    JOIN v ON a.vendor_name = v.vendor_name
    GROUP BY CASE WHEN v.rn <= 50 THEN 'Top 50 vendors' ELSE 'All others' END
    ORDER BY MIN(v.rn)
""").pl()

fig = go.Figure(go.Bar(
    x=tier_amend["tier"].to_list(), y=tier_amend["amend_rate"].to_list(),
    marker_color=['#D04040', '#4878A8'],
    text=[f"{v}%" for v in tier_amend["amend_rate"].to_list()],
    textposition="outside", cliponaxis=False))
fig.update_layout(title="Top vendors have nearly double the amendment rate",
    yaxis_title="Amendment rate (%)", template="plotly_white", height=400, margin=dict(t=60))
fig.show()

In [87]:
# Department dependency - which departments rely most on a single vendor?
con.sql("""
    WITH dept_vendor AS (
        SELECT owner_org_title, vendor_name, SUM(val) as vendor_total,
               SUM(SUM(val)) OVER (PARTITION BY owner_org_title) as dept_total,
               ROW_NUMBER() OVER (PARTITION BY owner_org_title ORDER BY SUM(val) DESC) as rn
        FROM analysis WHERE vendor_name IS NOT NULL AND val > 0
        GROUP BY owner_org_title, vendor_name
    )
    SELECT SPLIT_PART(owner_org_title, ' | ', 1) as department,
           vendor_name as top_vendor,
           ROUND(dept_total/1e9, 2) as dept_total_B,
           ROUND(vendor_total/1e9, 2) as vendor_total_B,
           ROUND(100.0 * vendor_total / dept_total, 1) as dependency_pct
    FROM dept_vendor
    WHERE rn = 1 AND dept_total > 1e9
    ORDER BY dependency_pct DESC
    LIMIT 10
""").pl()

department,top_vendor,dept_total_B,vendor_total_B,dependency_pct
str,str,f64,f64,f64
"""Department of Housing, Infrast…","""GROUPE SIGNATURE SUR LE""",25.05,10.06,40.1
"""Canadian Space Agency""","""MDA SYSTEMS LTD.""",40.29,15.7,39.0
"""Fisheries and Oceans Canada""","""VANCOUVER SHIPYARDS CO LTD""",63.51,24.52,38.6
"""Treasury Board of Canada Secre…","""Sun Life Assurance Company of …",5.71,2.19,38.3
"""Canada Border Services Agency""","""Deloitte Inc""",14.02,4.36,31.1
"""Employment and Social Developm…","""D+H CORPORATION""",34.93,10.05,28.8
"""Statistics Canada""","""MICROSOFT CANADA INC.""",1.2,0.33,27.7
"""Health Canada""","""SC2.0 STEPPED CARE SOLUTIONS I…",6.28,1.57,25.0
"""Elections Canada""","""KYNDRYL CANADA LTEE""",2.51,0.6,23.8


### Insight 3: what we found and what to do about it

**The pattern**: The top 50 vendors (out of 126,000+) hold ~55% of all spending. Their amendment rate is ~38% - nearly double the ~21% for all other vendors. 18 out of 97 departments have more than 30% of their spend with a single vendor.

**Why it matters**: Vendor concentration creates dependency risk (what happens if a key vendor fails?), reduces negotiating leverage, and - critically - the top vendors' higher amendment rate means the biggest contracts are also the ones most likely to grow beyond their original scope. This connects directly to Insight 2: vendor lock-in amplifies the amendment problem.

**The self-reinforcing cycle**: Q4 pressure (Insight 1) drives rushed awards. Those contracts get amended heavily (Insight 2). And the biggest vendors benefit the most from both patterns (Insight 3), concentrating more spending with fewer firms over time.

**What to do**:
- Map vendor dependency for critical services - flag departments where one vendor holds >30% of spend.
- Diversify the vendor base for recurring contract categories.
- Monitor whether top-vendor contracts are growing disproportionately through amendments.
- Consider vendor name normalization as a data quality initiative - true concentration is likely even higher than reported.

**Caveat**: Vendor names are not normalized. The same vendor appears under multiple spellings (e.g., "Canadian Corps of Commissionaires" has 10+ variants). True vendor concentration is likely higher than these numbers suggest.

---

## Connecting the dots

These three insights aren't separate problems. They're a self-reinforcing cycle.

**Q4 pressure** (Insight 1) drives a rush of contract awards and amendments at fiscal year-end. **Amendment growth** (Insight 2) expands contracts far beyond their original scope - services at 31.8%, with some contracts growing by thousands of percent. **Vendor concentration** (Insight 3) means the biggest vendors benefit the most from this cycle, with amendment rates nearly double the rest of the market (~38% vs ~21%).

The cycle works like this: year-end budget pressure creates rushed awards. Those awards go disproportionately to established vendors. Those vendors' contracts then grow through amendments, concentrating even more spending with the same firms. The result is a procurement system where the competitive process used for the original award becomes less meaningful over time.

## Federal benchmarks

| Metric | Federal benchmark |
|---|---|
| Q4 contract count vs Q1-Q3 average | +38% |
| Q4 construction value multiplier (post-2019) | 5.84x |
| Amendment rate (post-2019) | ~25% |
| Services amendment rate | 31.8% |
| Top 50 vendor share of total spend | ~55% |
| Top 50 vendor amendment rate | ~38% (vs ~21% for others) |
| Departments with >30% single-vendor dependency | 18 out of 97 |
| Q4 amendment rate vs Q1-Q3 | 25.7% vs 22.2% |

## What I'd investigate next

1. **Do Q4 contracts get amended more than Q1-Q3 contracts?** We showed amendments are more common *in* Q4, but are contracts *awarded* in Q4 more likely to be amended later?
2. **Which specific contract descriptions drive the Q4 volume surge?** Are certain categories (e.g., temporary help, consulting) disproportionately rushed at year-end?
3. **Vendor name normalization** - fuzzy matching vendor names would reveal the true concentration, which is likely even more extreme than reported here.
4. **Do top vendors bid low and grow through amendments?** Comparing original award values to final values by vendor tier could reveal whether strategic low-bidding is a pattern.